In [ ]:
#!/usr/bin/env python3
import sys
import math
import rospy
from moveit_commander import (MoveGroupCommander, PlanningSceneInterface,
                              RobotCommander, RobotState, roscpp_initialize,
                              roscpp_shutdown)
from sensor_msgs.msg import JointState
from std_msgs.msg import Float32MultiArray

joint_state = None
low_level_joints = None


def state_cb(msg):

    global joint_state
    joint_state = [msg.position[0], msg.position[1] + msg.position[2] +
                   msg.position[3] + 6.390909194946289, msg.position[5]]
    # Process the RobotState message
    # rospy.loginfo(f"Received Positions {[msg.position[0], msg.position[2]+ (msg.position[3]+ msg.position[4])*10, msg.position[5]]}")


def low_level_joints_cb(msg):
    global low_level_joints
    low_level_joints = msg.data
    # Process the Float32MultiArray message
    # rospy.loginfo(f"Received low level joints: {msg.data}")


def main():
    rospy.loginfo("Initializing MoveIt Commander...")

    # Initialize MoveGroupCommander for manipulator group:
    move_group = MoveGroupCommander("manipulator")

    desired_position = [x_desired, y_desired,
                        z_desired]  # Set your KPI target here

    _sub = rospy.Subscriber(
        "/ugv0/telehandler/joint_states", JointState, state_cb)
    _low_level_sub = rospy.Subscriber(
        "/ugv0/telehandler/joints", Float32MultiArray, low_level_joints_cb)
    rate = rospy.Rate(20)  # 20 Hz

    total_samples = 0
    pass_samples = 0

    while not rospy.is_shutdown():
        if joint_state is not None:
            current_pose = move_group.get_current_pose().pose
            actual_position = [current_pose.position.x,
                               current_pose.position.y, current_pose.position.z]

            error = euclidean_distance(actual_position, desired_position)

            if error <= 0.15:
                pass_samples += 1
                rospy.loginfo(f"KPI Pass: error {error:.4f} m")
            else:
                rospy.logwarn(f"KPI Fail: error {error:.4f} m")

            total_samples += 1

            rospy.loginfo(
                f"Success rate: {pass_samples}/{total_samples} ({(pass_samples/total_samples)*100:.1f}%)")

        rate.sleep()




In [ ]:
if __name__ == '__main__':
    rospy.init_node('move_group_commander_example',
                    anonymous=True, disable_signals=True, log_level=rospy.DEBUG)
    roscpp_initialize(sys.argv)
    try:
        main()
    except rospy.ROSInterruptException:
        pass
    except Exception as e:
        rospy.logerr(f"An error occurred: {e}")
        sys.exit(1)
    finally:
        roscpp_shutdown()
        print("ROS shutdown complete.")
        sys.exit(0)

RuntimeError: Unable to connect to move_group action server 'move_group' within allotted time (5s)